# EDW REST client (`geoREST.RESTesri.edw`) — usage examples

Walks through every public function in `src/geoREST/RESTesri/edw.py` against the live USFS EDW ArcGIS REST services:

- `search_edw_services` — find services by keyword/theme
- `get_service_info` — service-level metadata (layers, spatial ref)
- `get_layer_info` — layer-level metadata (fields, capabilities)
- `get_layer_metadata` — FGDC/ISO metadata (abstract, field definitions, domains)
- `query_features` — attribute + spatial feature queries
- `query_features_with_pagination` — queries beyond the 2000-record server cap
- `query_features_analytic` — SQL window functions (RANK, SUM, LAG, ...)
- `top_n_per_group` — convenience wrapper for "top N per group" queries

## Setup

This notebook needs `geoREST` installed:

```
pip install georest
```

or, from a clone of this repository, `pip install -e .` from the repo root.


In [ ]:
from geoREST.RESTesri.edw import (
    search_edw_services,
    get_service_info,
    get_layer_info,
    get_layer_metadata,
    query_features,
    query_features_with_pagination,
    query_features_analytic,
    top_n_per_group,
)

## 1. `search_edw_services` — find services by keyword or theme

Matches on service name, theme description, and a keyword-alias table (e.g. "riparian" pulls in inland-waters/hydro services even though the word never appears in a service name).

In [ ]:
# Plain keyword search
fire_services = search_edw_services("fire")
for s in fire_services[:5]:
    print(s["name"], "-", s["theme"])

In [ ]:
# Keyword-alias expansion: "riparian" isn't in any service name, but resolves
# to inland_waters/hydro/watershed/aquatic services via _KEYWORD_ALIASES
search_edw_services("riparian")

In [ ]:
# Filter by theme only (no keyword) — every transportation-themed service
search_edw_services(theme="transportation")

## 2. `get_service_info` — service-level metadata

Returns description, spatial reference, extent, and the list of layers in a MapServer.

In [ ]:
service_name = "EDW_MTBS_01"
info = get_service_info(service_name)

print("Service description (truncated at 200 characters):")
print(info["description"][:200])
print("# of layers in service:", len(info["layers"]))

print("Info on first 5 layers:")
for lyr in info["layers"][:5]:
    print(lyr["id"], lyr["name"])

## 3. `get_layer_info` — layer-level metadata

Fields, geometry type, and which operations/analytics the layer supports (`capabilities`, `advancedQueryCapabilities`).

In [ ]:
layer_id = 63  # "Burned Area Boundaries (All Years)"
layer_info = get_layer_info(service_name, layer_id)
print(layer_info["name"], layer_info["geometryType"])
print("capabilities:", layer_info["capabilities"])
[f["name"] for f in layer_info["fields"]][:10]

## 4. `get_layer_metadata` — FGDC/ISO metadata document

A separate document from `get_layer_info` — carries the dataset abstract/purpose and per-field *definitions* (what a field actually means), plus optional coded-value/range domains.

In [ ]:
meta = get_layer_metadata(service_name, layer_id, include_domains=True)
print(meta["title"])
print(meta["abstract"][:300])
print("keywords:", meta["keywords"][:5])

# Field definitions (only some fields are documented by the data provider)
for attr in meta["attributes"]:
    print(attr["name"], "->", attr["definition"] or "(undocumented)")

## 5. `query_features` — attribute and spatial queries

Returns a GeoJSON `FeatureCollection`. `where` filters by attributes; `geometry`/`geometry_type`/`spatial_rel` add a spatial filter.

In [ ]:
# Attribute-only query: a single named fire
cameron_peak = query_features(
    service_name, layer_id,
    where="fire_name = 'CAMERON PEAK'",
    out_fields="fire_name,ig_date,acres",
)
cameron_peak["features"]

In [ ]:
# return_count_only avoids pulling geometry/attributes when you only need a number
query_features(service_name, layer_id, where="acres > 100000", return_count_only=True)

In [ ]:
# Spatial query: use a forest boundary as the AOI to clip another layer.
# Grab the Idaho Panhandle NF boundary — a 161-part multipolygon with holes.
# query_features automatically falls back to an ID-based fetch for AOIs this
# geometrically complex (see the Note in its docstring), so this just works.
ipnf = query_features(
    "EDW_ForestSystemBoundaries_01", 0,
    where="FORESTORGCODE='0104'",
)
aoi_geom = ipnf["features"][0]["geometry"]

# ...then query fires intersecting it (geometry can be a GeoJSON dict, Esri JSON, or a bbox string)
ipnf_fires = query_features(
    service_name, layer_id,
    geometry=aoi_geom, geometry_type="esriGeometryPolygon",
    out_fields="fire_name,year,acres",
)
print(len(ipnf_fires["features"]), "fires intersecting the IPNF boundary")
for p in sorted((f["properties"] for f in ipnf_fires["features"]), key=lambda p: p["year"])[-5:]:
    print(f"{p['fire_name']} ({p['year']}): {p['acres']:,.0f} acres")

In [ ]:
# A plain bbox string also works as the geometry filter
bbox_fires = query_features(
    service_name, layer_id,
    geometry="-116.5,47.5,-116.0,48.0", geometry_type="esriGeometryEnvelope",
    out_fields="fire_name,year,acres", max_features=5,
)
for f in bbox_fires["features"]:
    p = f["properties"]
    print(f"{p['fire_name']} ({p['year']}): {p['acres']:,.0f} acres")

## 6. `query_features_with_pagination` — beyond the 2000-record server cap

Same signature as `query_features`, but `max_features` can exceed `_MAX_RECORD_COUNT`; it pages through `resultOffset` automatically. Using the point layer (fire ignition locations) here rather than the polygon layer — the EDW server 500s on exactly-2000-row pages of heavy polygon geometry, a server-side quirk unrelated to this function.

In [ ]:
all_fires = query_features_with_pagination(
    service_name, 62,  # "Fire Occurrence Locations (All Years)" — points, not polygons
    out_fields="fire_name,ig_date,acres",
    max_features=4000,
)
len(all_fires["features"])

## 7. `query_features_analytic` — SQL window functions

Runs ArcGIS's `queryAnalytic` operation (RANK, SUM, LAG/LEAD, PERCENTILE_CONT, ...). Unlike `query_features`, rows aren't collapsed — each analytic value is appended as a new field on its source feature.

Note: ArcGIS ignores whatever `out_name` you request for a `RANK` analytic and always names the computed field `rank_expr0` — filter on that name in `analytic_where`, not your requested `out_name`.

In [ ]:
# Rank fires by acreage within each ignition year, keep only the #1 fire per year
biggest_per_year = query_features_analytic(
    service_name, layer_id,
    out_analytics=[{
        "type": "RANK",
        "field": "acres",
        "order_by": "acres DESC",
        "out_name": "acres_rank",
    }],
    partition_by="year",
    analytic_where="rank_expr0 = 1",
    out_fields="fire_name,year,acres",
    return_geometry=False,
)
sorted(
    (f["properties"] for f in biggest_per_year["features"]),
    key=lambda p: p["year"],
)[-5:]

**Other analytic types:** the Esri `queryAnalytic` spec also defines `SUM`, `AVG`, `MIN`, `MAX`, `COUNT`, `STDDEV`, `VAR`, `LAG`, `LEAD`, `NTILE`, `FIRST_VALUE`, `LAST_VALUE`, `PERCENTILE_CONT`, and `PERCENTILE_DISC` — but this EDW server only supports the ranking-family functions in practice: `RANK` (above), `DENSE_RANK`, `ROW_NUMBER`, `PERCENT_RANK`, and `CUME_DIST`. The rest fail with `"Unable to complete operation"` (or an explicit `"not supported"` for the percentile functions), regardless of how they're called.

`ROW_NUMBER` is a useful contrast to `RANK`: `RANK` gives tied values the *same* rank, so a tie for smallest/largest fire in a given year produces multiple rows for that year (e.g. 2024 has two fires tied at exactly 501 acres). `ROW_NUMBER` breaks ties arbitrarily but deterministically, guaranteeing exactly one row per partition — useful when you need a strict 1-per-group result regardless of ties.

In [ ]:
# Smallest fire per year via ROW_NUMBER — exactly one row per year, even in years
# (like 2024) where two fires are tied at exactly the same acreage
smallest_per_year = query_features_analytic(
    service_name, layer_id,
    out_analytics=[{
        "type": "ROW_NUMBER",
        "order_by": "acres ASC",
        "out_name": "row_num",
    }],
    partition_by="year",
    analytic_where="row_number_expr0 = 1",
    out_fields="fire_name,year,acres",
    return_geometry=False,
)
print(len(smallest_per_year["features"]), "features (one per year)")
for f in sorted((f["properties"] for f in smallest_per_year["features"]), key=lambda p: p["year"])[-5:]:
    print(f)

## 8. `top_n_per_group` — convenience wrapper

Same query as above, without hand-rolling the RANK analytic. `descending=False` ranks smallest-first instead.

In [ ]:
biggest_by_year = top_n_per_group(
    service_name, layer_id,
    field="acres",
    group_by="year",
    descending=True,
    n=1,
    out_fields="fire_name,year,acres",
)

sorted(
    (f["properties"] for f in biggest_by_year["features"]),
    key=lambda p: p["year"],
)[-5:]

Smallest fire by year. Note this function will silently return more than N entries per group if there is a tie. (See 2024 duplicate below). If you need a single entry per group, use query_analytic() with ROW_NUMBER.

In [ ]:
smallest_by_year = top_n_per_group(
    service_name, layer_id,
    field="acres",
    group_by="year",
    descending=False, # set to false for smallest first
    n=1,
    out_fields="fire_name,year,acres",
)

sorted(
    (f["properties"] for f in smallest_by_year["features"]),
    key=lambda p: p["year"],
)[-5:]

Get the largest fire that started that started in each month across all years.

In [ ]:
biggest_by_month = top_n_per_group(
    service_name, layer_id,
    field="acres",
    group_by="startmonth",
    descending=True,
    n=1,
    out_fields="fire_name,year,startmonth,acres",
)

sorted(
    (f["properties"] for f in biggest_by_month["features"]),
    key=lambda p: p["startmonth"],
)

Look at the 10 largest fires just in 2017. 

In [ ]:
largest_2017 = top_n_per_group(
    service_name, layer_id,
    field="acres",
    group_by="year",
    descending=True,
    n=10,
    where="year=2017",
    out_fields="fire_name,year,startmonth,acres",
)

sorted(
    (f["properties"] for f in largest_2017["features"]),
    key=lambda p: p["year"],
)